# Cross-Validated Accuracy Summary
Loads saved prediction pickles and computes mean CV accuracy ± SEM and 95% CI across mice.

Each prediction pickle contains one dataclass object per (mouse, stride, phase) combination.
Each object has `.cv_acc` — an array of per-fold balanced accuracies (typically 10 folds).

**Pipeline**: per-fold acc → mean across folds per mouse → mean/SEM/CI across mice.

In [29]:
import pickle
import numpy as np
import pandas as pd
from scipy import stats

# Register backward-compat alias so pickles saved under the old
# module path (Analysis.Characterisation_v2.*) can be unpickled
import apa_analysis

In [ ]:
def cv_accuracy_summary(pred_path, stride, phase_filter, label=""):
    """
    Load a prediction pickle and return per-mouse mean CV accuracy,
    overall mean, SEM, and 95% CI.
    
    Works with:
      - PCAPredictionData (Main.py): .phase is a tuple e.g. ('APA2','Wash2'), .stride is int
      - RegressionPredicitonData (CompareConditions): .phase is a str e.g. 'apa', .stride is int,
        .conditions is a list e.g. ['APAChar_LowHigh','APAChar_HighLow']
    
    Parameters
    ----------
    pred_path : str
        Path to the .pkl file.
    stride : int
        Stride number to filter on (e.g. -1, -2, -3).
    phase_filter
        For Main.py models: tuple like ('APA2', 'Wash2').
        For CompareConditions models: str like 'apa', or None to skip phase filtering.
        For CompareConditions models with conditions filter: dict like
            {'phase': 'apa', 'conditions': ['APAChar_LowHigh', 'APAChar_HighLow']}
    label : str
        Display label for this model.
    """
    with open(pred_path, 'rb') as f:
        preds = pickle.load(f)
    
    # Parse phase_filter
    conditions_filter = None
    if isinstance(phase_filter, dict):
        conditions_filter = phase_filter.get('conditions')
        phase_filter = phase_filter.get('phase')
    
    # Filter predictions
    filtered = []
    for p in preds:
        if p.stride != stride:
            continue
        if phase_filter is not None:
            if isinstance(phase_filter, tuple) and isinstance(p.phase, tuple):
                if p.phase != phase_filter:
                    continue
            elif isinstance(phase_filter, str) and isinstance(p.phase, str):
                if p.phase != phase_filter:
                    continue
            elif p.phase != phase_filter:
                continue
        if conditions_filter is not None and hasattr(p, 'conditions'):
            if sorted(p.conditions) != sorted(conditions_filter):
                continue
        if p.cv_acc is None:
            continue
        filtered.append(p)
    
    if not filtered:
        print(f"[{label}] No predictions found for stride={stride}, phase={phase_filter}")
        return None
    
    # Per-mouse mean CV accuracy (mean across folds)
    mouse_ids = [p.mouse_id for p in filtered]
    mice_mean_cv = np.array([np.mean(p.cv_acc) for p in filtered])
    n = len(mice_mean_cv)
    
    overall_mean = np.mean(mice_mean_cv)
    sem = np.std(mice_mean_cv, ddof=1) / np.sqrt(n)
    
    # 95% CI using t-distribution
    t_crit = stats.t.ppf(0.975, df=n - 1)
    ci_low = overall_mean - t_crit * sem
    ci_high = overall_mean + t_crit * sem
    
    per_mouse_df = pd.DataFrame({
        'MouseID': mouse_ids,
        'MeanCVAcc': mice_mean_cv
    }).set_index('MouseID')
    
    summary = {
        'label': label,
        'n_mice': n,
        'mean': overall_mean,
        'sem': sem,
        'ci_95_low': ci_low,
        'ci_95_high': ci_high,
        'per_mouse': per_mouse_df
    }
    
    print(f"--- {label} (n={n}, stride={stride}) ---")
    print(f"  Mean: {overall_mean:.4f}  SEM: {sem:.4f}  95% CI: [{ci_low:.4f}, {ci_high:.4f}]")
    print(per_mouse_df.to_string())
    print()
    return summary


def summarise_all(results):
    """Collect all non-None results into a summary DataFrame."""
    valid = [r for r in results if r is not None]
    return pd.DataFrame([{
        'Model': r['label'],
        'N': r['n_mice'],
        'Mean CV Acc': round(r['mean'], 4),
        'SEM': round(r['sem'], 4),
        '95% CI': f"[{r['ci_95_low']:.4f}, {r['ci_95_high']:.4f}]"
    } for r in valid])

In [31]:
PHASES = ('APA2', 'Wash2')
STRIDES = [-3, -2, -1]  # all strides to report

---
## Main within-condition models (from Main.py)
Each pickle contains PCAPredictionData objects for all (mouse, stride, phase) combos.

### Main Low-High model (all PCs)

In [32]:
lh_main_path = r"H:\Characterisation_v2\LH_corrections_res_-3-2-1_APA2Wash2\APAChar_LowHigh_Extended\MultiFeaturePredictions\pca_predictions_APAChar_LowHigh.pkl"

lh_main = []
for s in STRIDES:
    lh_main.append(cv_accuracy_summary(lh_main_path, s, PHASES, label=f"Main LH (stride {s})"))

--- Main LH (stride -3) (n=8, stride=-3) ---
  Mean: 0.7603  SEM: 0.0285  95% CI: [0.6930, 0.8277]
         MeanCVAcc
MouseID           
1035243   0.805000
1035244   0.795000
1035245   0.729167
1035246   0.698333
1035250   0.902500
1035297   0.729167
1035299   0.635833
1035301   0.787500

--- Main LH (stride -2) (n=8, stride=-2) ---
  Mean: 0.7971  SEM: 0.0293  95% CI: [0.7279, 0.8663]
         MeanCVAcc
MouseID           
1035243   0.823333
1035244   0.855833
1035245   0.832500
1035246   0.852500
1035250   0.862500
1035297   0.712500
1035299   0.630000
1035301   0.807500

--- Main LH (stride -1) (n=8, stride=-1) ---
  Mean: 0.8205  SEM: 0.0333  95% CI: [0.7417, 0.8994]
         MeanCVAcc
MouseID           
1035243   0.780833
1035244   0.787500
1035245   0.935000
1035246   0.840000
1035250   0.983333
1035297   0.762500
1035299   0.702500
1035301   0.772500



### Top PCs Low-High model

In [33]:
lh_top3_path = r"H:\Characterisation_v2\LH_corrections_res_-3-2-1_APA2Wash2\APAChar_LowHigh_Extended\MultiFeaturePredictions\TopPCs\pca_predictions_top3_APAChar_LowHigh.pkl"

# Top PCs model was only run for stride -1
lh_top3 = cv_accuracy_summary(lh_top3_path, -1, PHASES, label="Top PCs LH (stride -1)")

--- Top PCs LH (stride -1) (n=8, stride=-1) ---
  Mean: 0.7335  SEM: 0.0355  95% CI: [0.6496, 0.8174]
         MeanCVAcc
MouseID           
1035243   0.638333
1035244   0.787500
1035245   0.724167
1035246   0.738333
1035250   0.904167
1035297   0.700000
1035299   0.578333
1035301   0.797500



### High-Low predictions (using LH model)

In [34]:
hl_lhmodel_path = r"H:\Characterisation_v2\HL_test_LHmodels_LhWnrm_res_-3-2-1_APA2Wash2\APAChar_HighLow_Extended\MultiFeaturePredictions\pca_predictions_APAChar_HighLow.pkl"

hl_lhmodel = []
for s in STRIDES:
    hl_lhmodel.append(cv_accuracy_summary(hl_lhmodel_path, s, PHASES, label=f"Main HL (stride {s})"))

--- Main HL (stride -3) (n=6, stride=-3) ---
  Mean: 0.5110  SEM: 0.0362  95% CI: [0.4180, 0.6040]
         MeanCVAcc
MouseID           
1035243   0.514167
1035244   0.658333
1035245   0.400000
1035246   0.453571
1035250   0.490000
1035301   0.550000

--- Main HL (stride -2) (n=6, stride=-2) ---
  Mean: 0.4474  SEM: 0.0750  95% CI: [0.2546, 0.6403]
         MeanCVAcc
MouseID           
1035243   0.345000
1035244   0.731389
1035245   0.248333
1035246   0.592222
1035250   0.316667
1035301   0.451071

--- Main HL (stride -1) (n=6, stride=-1) ---
  Mean: 0.4283  SEM: 0.0532  95% CI: [0.2916, 0.5651]
         MeanCVAcc
MouseID           
1035243   0.367778
1035244   0.600000
1035245   0.273889
1035246   0.529444
1035250   0.488889
1035301   0.310000



### Main High-Low model

In [35]:
hl_main_path = r"H:\Characterisation_v2\HL_LHpcsonly_LhWnrm_res_-3-2-1_APA2Wash2\APAChar_HighLow_Extended\MultiFeaturePredictions\pca_predictions_APAChar_HighLow.pkl"

hl_main = []
for s in STRIDES:
    hl_main.append(cv_accuracy_summary(hl_main_path, s, PHASES, label=f"Main HL (stride {s})"))

--- Main HL (stride -3) (n=6, stride=-3) ---
  Mean: 0.6360  SEM: 0.0476  95% CI: [0.5137, 0.7582]
         MeanCVAcc
MouseID           
1035243   0.715833
1035244   0.525000
1035245   0.675000
1035246   0.462500
1035250   0.675000
1035301   0.762500

--- Main HL (stride -2) (n=6, stride=-2) ---
  Mean: 0.7589  SEM: 0.0299  95% CI: [0.6821, 0.8356]
         MeanCVAcc
MouseID           
1035243   0.809167
1035244   0.702500
1035245   0.691667
1035246   0.705000
1035250   0.770000
1035301   0.875000

--- Main HL (stride -1) (n=6, stride=-1) ---
  Mean: 0.7267  SEM: 0.0609  95% CI: [0.5701, 0.8833]
         MeanCVAcc
MouseID           
1035243   0.937500
1035244   0.632500
1035245   0.630000
1035246   0.599167
1035250   0.665000
1035301   0.895833



### Main Low-Mid model

In [36]:
lm_main_path = r"H:\Characterisation_v2\LM_LHpcsonly_res_-3-2-1_APA2Wash2\APAChar_LowMid_Extended\MultiFeaturePredictions\pca_predictions_APAChar_LowMid.pkl"

lm_main = []
for s in STRIDES:
    lm_main.append(cv_accuracy_summary(lm_main_path, s, PHASES, label=f"Main LM (stride {s})"))

--- Main LM (stride -3) (n=6, stride=-3) ---
  Mean: 0.7547  SEM: 0.0384  95% CI: [0.6560, 0.8534]
         MeanCVAcc
MouseID           
1035243   0.900000
1035245   0.840000
1035246   0.733333
1035297   0.670833
1035299   0.708333
1035301   0.675833

--- Main LM (stride -2) (n=6, stride=-2) ---
  Mean: 0.8014  SEM: 0.0324  95% CI: [0.7180, 0.8848]
         MeanCVAcc
MouseID           
1035243   0.933333
1035245   0.843333
1035246   0.791667
1035297   0.715000
1035299   0.732500
1035301   0.792500

--- Main LM (stride -1) (n=6, stride=-1) ---
  Mean: 0.7785  SEM: 0.0336  95% CI: [0.6920, 0.8650]
         MeanCVAcc
MouseID           
1035243   0.858333
1035245   0.805833
1035246   0.758333
1035297   0.789167
1035299   0.625833
1035301   0.833333



---
## Cross-condition comparison models (from CompareConditions_Regression.py)
These pickles contain RegressionPredicitonData objects. Filter by `.conditions` to pick the right pair.

**Note**: These pickles are saved after running `CompareConditions_Regression.py` (added `pickle.dump` at end of `run()`).
If the pickle doesn't exist yet, re-run that script first.

### LH vs HL comparison

In [37]:
lh_vs_hl_path = r"H:\Characterisation_v2\Compare_CORRECTIONS_LowHigh_vs_HighLow_regression_chosen_pcs\reg_predictions_LowHigh_HighLow.pkl"

lh_vs_hl = []
for s in STRIDES:
    lh_vs_hl.append(cv_accuracy_summary(
        lh_vs_hl_path, s,
        {'phase': 'apa', 'conditions': ['APAChar_LowHigh', 'APAChar_HighLow']},
        label=f"LH vs HL (stride {s})"
    ))

--- LH vs HL (stride -3) (n=6, stride=-3) ---
  Mean: 0.5817  SEM: 0.0261  95% CI: [0.5147, 0.6486]
         MeanCVAcc
MouseID           
1035244   0.540000
1035245   0.630000
1035250   0.661667
1035243   0.485000
1035301   0.570833
1035246   0.602500

--- LH vs HL (stride -2) (n=6, stride=-2) ---
  Mean: 0.6396  SEM: 0.0249  95% CI: [0.5757, 0.7035]
         MeanCVAcc
MouseID           
1035244     0.6500
1035245     0.6100
1035250     0.7500
1035243     0.5950
1035301     0.5825
1035246     0.6500

--- LH vs HL (stride -1) (n=6, stride=-1) ---
  Mean: 0.6300  SEM: 0.0354  95% CI: [0.5389, 0.7211]
         MeanCVAcc
MouseID           
1035244   0.630000
1035245   0.635000
1035250   0.793333
1035243   0.552500
1035301   0.564167
1035246   0.605000



### LH vs LM comparison

In [38]:
lh_vs_lm_path = r"H:\Characterisation_v2\Compare_CORRECTIONS_LowHigh_vs_LowMid_regression_chosen_pcs\reg_predictions_LowHigh_LowMid.pkl"

lh_vs_lm = []
for s in STRIDES:
    lh_vs_lm.append(cv_accuracy_summary(
        lh_vs_lm_path, s,
        {'phase': 'apa', 'conditions': ['APAChar_LowHigh', 'APAChar_LowMid']},
        label=f"LH vs LM (stride {s})"
    ))

--- LH vs LM (stride -3) (n=6, stride=-3) ---
  Mean: 0.5064  SEM: 0.0224  95% CI: [0.4487, 0.5641]
         MeanCVAcc
MouseID           
1035299   0.552500
1035297   0.465000
1035245   0.482500
1035243   0.429167
1035301   0.547500
1035246   0.561667

--- LH vs LM (stride -2) (n=6, stride=-2) ---
  Mean: 0.5315  SEM: 0.0387  95% CI: [0.4320, 0.6311]
         MeanCVAcc
MouseID           
1035299   0.502500
1035297   0.400000
1035245   0.542500
1035243   0.598333
1035301   0.670000
1035246   0.475833

--- LH vs LM (stride -1) (n=6, stride=-1) ---
  Mean: 0.5431  SEM: 0.0465  95% CI: [0.4234, 0.6627]
         MeanCVAcc
MouseID           
1035299   0.515000
1035297   0.485000
1035245   0.447500
1035243   0.535833
1035301   0.767500
1035246   0.507500



### LM vs HL comparison

In [39]:
lm_vs_hl_path = r"H:\Characterisation_v2\Compare_CORRECTIONS_LowMid_vs_HighLow_regression_chosen_pcs\reg_predictions_LowMid_HighLow.pkl"

lm_vs_hl = []
for s in STRIDES:
    lm_vs_hl.append(cv_accuracy_summary(
        lm_vs_hl_path, s,
        {'phase': 'apa', 'conditions': ['APAChar_LowMid', 'APAChar_HighLow']},
        label=f"LM vs HL (stride {s})"
    ))

--- LM vs HL (stride -3) (n=4, stride=-3) ---
  Mean: 0.5956  SEM: 0.0276  95% CI: [0.5079, 0.6834]
         MeanCVAcc
MouseID           
1035246   0.662500
1035301   0.610833
1035243   0.531667
1035245   0.577500

--- LM vs HL (stride -2) (n=4, stride=-2) ---
  Mean: 0.6602  SEM: 0.0420  95% CI: [0.5266, 0.7938]
         MeanCVAcc
MouseID           
1035246   0.701667
1035301   0.645000
1035243   0.744167
1035245   0.550000

--- LM vs HL (stride -1) (n=4, stride=-1) ---
  Mean: 0.6569  SEM: 0.0370  95% CI: [0.5391, 0.7747]
         MeanCVAcc
MouseID           
1035246   0.657500
1035301   0.743333
1035243   0.664167
1035245   0.562500



---
## Summary table

In [40]:
all_results = (
    lh_main + [lh_top3] +
    hl_lhmodel + hl_main + lm_main +
    lh_vs_hl + lh_vs_lm + lm_vs_hl
)

summary_df = summarise_all(all_results)
summary_df

,Model,N,Mean CV Acc,SEM,95% CI
0,Main LH (stride -3),8,0.7603,0.0285,"[0.6930, 0.8277]"
1,Main LH (stride -2),8,0.7971,0.0293,"[0.7279, 0.8663]"
2,Main LH (stride -1),8,0.8205,0.0333,"[0.7417, 0.8994]"
3,Top PCs LH (stride -1),8,0.7335,0.0355,"[0.6496, 0.8174]"
4,Main HL (stride -3),6,0.5110,0.0362,"[0.4180, 0.6040]"
5,Main HL (stride -2),6,0.4474,0.0750,"[0.2546, 0.6403]"
6,Main HL (stride -1),6,0.4283,0.0532,"[0.2916, 0.5651]"
7,Main HL (stride -3),6,0.6360,0.0476,"[0.5137, 0.7582]"
8,Main HL (stride -2),6,0.7589,0.0299,"[0.6821, 0.8356]"
9,Main HL (stride -1),6,0.7267,0.0609,"[0.5701, 0.8833]"
